# Lab 19-12 · Faults, Scan Chains and Self-Test

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import itertools
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import dft

## Part 1 · Faults and tests

*Predict* the fault counts, undetectable faults and minimal test-set sizes for c17 and for $AB + \\bar{A}C + BC$.

In [2]:
c17 = dft.Circuit(["N1", "N2", "N3", "N6", "N7"],
                  [("N10", "NAND", ["N1", "N3"]), ("N11", "NAND", ["N3", "N6"]), ("N16", "NAND", ["N2", "N11"]),
                   ("N19", "NAND", ["N11", "N7"]), ("N22", "NAND", ["N10", "N16"]), ("N23", "NAND", ["N16", "N19"])],
                  ["N22", "N23"])
cons = dft.Circuit(["A", "B", "C"], [("nA", "NOT", ["A"]), ("g1", "AND", ["A", "B"]), ("g2", "AND", ["nA", "C"]),
                                     ("g3", "AND", ["B", "C"]), ("F", "OR", ["g1", "g2", "g3"])], ["F"])

def minimal_tests(c):
    ts = {f: s for f, s in c.test_sets().items() if s}
    for k in range(1, 8):
        for combo in itertools.combinations(c.all_vectors(), k):
            if all(s & set(combo) for s in ts.values()):
                return combo

for name, c in (("c17", c17), ("consensus", cons)):
    ts = c.test_sets()
    print(f"{name}: {len(ts)} faults, {len(dft.equivalence_classes(c))} classes, "
          f"{sum(1 for s in ts.values() if not s)} undetectable, minimal tests {minimal_tests(c)}")
print("undetectable in the consensus circuit:", [f for f, s in cons.test_sets().items() if not s])

c17: 34 faults, 21 classes, 0 undetectable, minimal tests ((0, 0, 1, 0, 0), (0, 1, 1, 0, 0), (1, 0, 0, 1, 1), (1, 1, 1, 1, 1))
consensus: 28 faults, 13 classes, 3 undetectable, minimal tests ((0, 0, 1), (0, 1, 0), (1, 0, 1), (1, 1, 0))
undetectable in the consensus circuit: [(('g3', 0), 0), (('g3', 1), 0), ('g3', 0)]


## Part 2 · Random patterns

*Predict* the shape of the coverage curve for a 10-input AND feeding an OR with an XOR of two other inputs.

In [3]:
names = [f"a{k}" for k in range(10)] + ["b0", "b1"]
wide = dft.Circuit(names, [("w", "AND", names[:10]), ("x", "XOR", ["b0", "b1"]), ("out", "OR", ["w", "x"])], ["out"])
ts = wide.test_sets()
print("faults detected by exactly 2 of 4096 vectors:", sum(1 for s in ts.values() if len(s) == 2), "of", len(ts))
for seed in range(5):
    plt.semilogx(100 * np.array(dft.random_coverage(wide, 20000, seed=seed)), lw=1)
plt.axvline(2 ** 11, ls="--", color="k")
plt.xlabel("random patterns"); plt.ylabel("coverage (%)"); plt.show()
print("patterns for 95 % confidence on one hard fault:", round(dft.patterns_for_detection(2 ** -11, 0.95)))

faults detected by exactly 2 of 4096 vectors: 21 of 30


patterns for 95 % confidence on one hard fault: 6134


## Part 3 · A scan chain in Verilog

*Predict* the clock-cycle count and which of the patterns 00, 5A, FF, 80 catch the stuck-at-0 on bit 2.

In [4]:
print("predicted cycles:", dft.scan_cycles(8, 1, 4))
print("patterns whose response bit 2 is 1:", [hex(p) for p in (0x00, 0x5A, 0xFF, 0x80) if ((p + 3) & 0xFF) >> 2 & 1])
if shutil.which("iverilog"):
    import os, tempfile
    sim = os.path.join(tempfile.mkdtemp(), "scan_sim.vvp")
    subprocess.run(["iverilog", "-g2012", "-o", sim, "scan_reg_tb.v", "scan_reg.v"], check=True)
    print(subprocess.run(["vvp", sim], capture_output=True, text=True).stdout)
else:
    print("Icarus Verilog not installed; the README lists the expected output")

predicted cycles: 44
patterns whose response bit 2 is 1: ['0x5a']
pattern 1 (5a) detects the stuck-at-0 on bit 2: 59 instead of 5d
4 patterns, 44 clock cycles, fault detected by 1 pattern(s)
TEST PASSED
scan_reg_tb.v:55: $finish called at 436000 (1ps)



## Part 4 · LFSRs and a MISR

*Predict* the periods and the aliasing fraction.

In [5]:
print("x^4 + x^3 + 1:", len(dft.lfsr_sequence([4, 3], 4)), "states;  x^4 + x^2 + 1:", len(dft.lfsr_sequence([4, 2], 4)), "states")
alias = sum(1 for e in itertools.product(range(16), repeat=3) if any(e) and dft.misr_signature(list(e), [4, 3], 4) == 0)
print("aliasing error streams:", alias, "of", 16 ** 3 - 1, "=", round(alias / (16 ** 3 - 1), 5), "; 2^-4 =", 2 ** -4)

x^4 + x^3 + 1: 15 states;  x^4 + x^2 + 1: 6 states
aliasing error streams: 255 of 4095 = 0.06227 ; 2^-4 = 0.0625


## Part 5 · Memory tests

*Predict* which faults MATS+ misses.

In [6]:
MATS_PLUS = [("any", ["w0"]), ("up", ["r0", "w1"]), ("down", ["r1", "w0"])]
n = 8
simple = [("saf", a, v) for a in range(n) for v in (0, 1)] + [("tf", a, r) for a in range(n) for r in (True, False)]
coupling = [("cfid", ag, vi, r, v) for ag in range(n) for vi in range(n) if ag != vi for r in (True, False) for v in (0, 1)]
for name, alg in (("March C-", dft.MARCH_C_MINUS), ("MATS+", MATS_PLUS)):
    miss_s = [f for f in simple if not dft.run_march(dft.FaultyMemory(n, [f]), alg)]
    miss_c = [f for f in coupling if not dft.run_march(dft.FaultyMemory(n, [f]), alg)]
    print(f"{name} ({dft.march_ops(alg)} ops/cell): misses {len(miss_s)} of {len(simple)} stuck-at/transition, "
          f"{len(miss_c)} of {len(coupling)} coupling; e.g. {miss_s[:2]}")

March C- (10 ops/cell): misses 0 of 32 stuck-at/transition, 0 of 224 coupling; e.g. []
MATS+ (5 ops/cell): misses 8 of 32 stuck-at/transition, 140 of 224 coupling; e.g. [('tf', 0, False), ('tf', 1, False)]


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).